# vLLM Configuration — A Learning Notebook

Goal: understand what every knob in our `run_*.sh` deployment scripts does — not by memorizing docs,
but by reading the actual code that consumes each flag, and redoing the math behind our real incidents.

**Ground rules of this notebook**
- Every claim is backed by source you can read: `~/projects/ai/vllm` and `~/projects/ai/vllm-ascend`,
  both synced to the exact build running in our containers (vllm `v0.18.0`, vllm-ascend branch `v4_v0.18.0_0412`).
- Code cells locate source by **symbol search**, not hardcoded line numbers, so they survive branch updates.
- All cells run on the Mac with stdlib only — no vLLM install, no NPU.

## Learning plan (≈6 sessions)

| # | Chapter | Done when you can… |
|---|---------|--------------------|
| 1 | The mental model | sketch a request's path from HTTP to streamed tokens, and explain continuous batching in 3 sentences |
| 2 | The memory model | compute KV-cache bytes/token for a model from its config, and explain our 0.85 → 0.90 fix and the 0.94 runtime OOM |
| 3 | Scheduler knobs | predict how the scheduler spends its token budget for given `max-num-seqs` / `max-num-batched-tokens`, and justify prefill `8192/4` vs decode `144/48` |
| 4 | Parallelism: TP / DP / EP / PP | say what each one shards, what communication it adds, and why our 1P2D design halves per-NPU expert memory |
| 5 | Our fleet, annotated | read any `run_*.sh` in this repo and explain every flag |
| 6 | Pointers for later | know where in the source to dig for MTP, w8a8 quant, cudagraph, FlashComm |

## 0. Setup

Two helpers used throughout:
- `show(repo, relpath, pattern, before=2, after=25)` — find `pattern` in a source file and print it with real line numbers (so you can jump there in an editor).
- `grep_src(repo, pattern, glob)` — list matches across a repo when you want to explore.

The cell below also checks that the checkouts are on the build we actually deploy.

In [1]:
import re, subprocess
from pathlib import Path

VLLM = Path.home() / "projects/ai/vllm"
VLLM_ASCEND = Path.home() / "projects/ai/vllm-ascend"
REPO = Path.home() / "projects/ai/ma-vllm-samples"

def _branch(repo):
    return subprocess.run(["git", "-C", str(repo), "describe", "--all", "--always"],
                          capture_output=True, text=True).stdout.strip()

print("vllm:        ", _branch(VLLM), "(deployed build is tag v0.18.0)")
print("vllm-ascend: ", _branch(VLLM_ASCEND), "(deployed build is branch v4_v0.18.0_0412)")

def show(repo, relpath, pattern, before=2, after=25):
    """Print the region of repo/relpath around the first line matching `pattern` (regex)."""
    path = Path(repo) / relpath
    lines = path.read_text(encoding="utf-8").splitlines()
    for i, line in enumerate(lines):
        if re.search(pattern, line):
            lo, hi = max(0, i - before), min(len(lines), i + after)
            print(f"--- {path}:{i + 1} ---")
            for n in range(lo, hi):
                print(f"{n + 1:5d}  {lines[n]}")
            return
    print(f"pattern {pattern!r} not found in {path} — the branch may have changed; try grep_src")

def grep_src(repo, pattern, glob="*.py", max_hits=20):
    hits = []
    for p in Path(repo).rglob(glob):
        if ".git" in p.parts:
            continue
        try:
            for i, line in enumerate(p.read_text(encoding="utf-8").splitlines()):
                if re.search(pattern, line):
                    hits.append(f"{p.relative_to(repo)}:{i + 1}: {line.strip()[:100]}")
                    if len(hits) >= max_hits:
                        print("\n".join(hits)); return
        except (UnicodeDecodeError, OSError):
            pass
    print("\n".join(hits) if hits else "no matches")

vllm:         tags/v0.18.0 (deployed build is tag v0.18.0)
vllm-ascend:  heads/v4_v0.18.0_0412 (deployed build is branch v4_v0.18.0_0412)


## 1. The mental model: one request's life

```
HTTP request ──> APIServer process (tokenize, build SamplingParams)
                    │  ZMQ
                    v
                 EngineCore process — an infinite loop:
                    1. scheduler picks WHICH requests advance this step, and by HOW MANY tokens
                    2. model executor runs ONE forward pass for that whole batch
                    3. sampled tokens stream back; finished requests leave, waiting ones join
```

Two ideas carry everything else:

**Continuous batching** — the batch is rebuilt *every step*. A request that just arrived can join the
very next forward pass alongside requests that are 500 tokens into generation. Nothing waits for a
\"batch to fill up\". This is why throughput knobs are per-*step* budgets (chapter 3), not batch sizes.

**Prefill vs decode are the same loop, different token counts** — a prompt being ingested contributes
many tokens to a step (compute-bound); a generating request contributes 1 (+ speculative) tokens
(memory-bandwidth-bound). One engine interleaves both — unless you split them across machines like our
PD-disaggregated layouts do.

Read the loop yourself — `step()` is genuinely short:

In [2]:
# The heart of the engine: schedule -> execute -> update.
show(VLLM, "vllm/v1/engine/core.py", r"def step\(self\)", after=30)

--- /Users/sean/projects/ai/vllm/vllm/v1/engine/core.py:378 ---
  376          self._iteration_index += 1
  377  
  378      def step(self) -> tuple[dict[int, EngineCoreOutputs], bool]:
  379          """Schedule, execute, and make output.
  380  
  381          Returns tuple of outputs and a flag indicating whether the model
  382          was executed.
  383          """
  384  
  385          # Check for any requests remaining in the scheduler - unfinished,
  386          # or finished and not yet removed from the batch.
  387          if not self.scheduler.has_requests():
  388              return {}, False
  389          scheduler_output = self.scheduler.schedule()
  390          future = self.model_executor.execute_model(scheduler_output, non_block=True)
  391          grammar_output = self.scheduler.get_grammar_bitmask(scheduler_output)
  392          with (
  393              self.log_error_detail(scheduler_output),
  394              self.log_iteration_details(scheduler_output

In [3]:
# Each DP rank runs this as its own process (the EngineCore_DP1 / pid=262 you see in our logs).
show(VLLM, "vllm/v1/engine/core.py", r"def run_engine_core", after=20)

--- /Users/sean/projects/ai/vllm/vllm/v1/engine/core.py:1029 ---
 1027  
 1028      @staticmethod
 1029      def run_engine_core(*args, dp_rank: int = 0, local_dp_rank: int = 0, **kwargs):
 1030          """Launch EngineCore busy loop in background process."""
 1031  
 1032          # Ensure we can serialize transformer config after spawning
 1033          maybe_register_config_serialize_by_value()
 1034  
 1035          engine_core: EngineCoreProc | None = None
 1036          signal_callback: SignalCallback | None = None
 1037          try:
 1038              vllm_config: VllmConfig = kwargs["vllm_config"]
 1039              parallel_config: ParallelConfig = vllm_config.parallel_config
 1040              data_parallel = parallel_config.data_parallel_size > 1 or dp_rank > 0
 1041              if data_parallel:
 1042                  parallel_config.data_parallel_rank_local = local_dp_rank
 1043                  process_title = f"EngineCore_DP{dp_rank}"
 1044              else:
 1045   

**Check yourself**
1. Why does a long prompt slow down *other* users' token latency in a shared engine? (Hint: one forward pass per step, shared by all.)
2. In our crash logs, the traceback came from `EngineCore_DP1 pid=262` while the API server was `pid=6`. Why are they different processes?

## 2. The memory model: where HBM goes

At startup vLLM splits each device's memory budget like this:

```
gpu_memory_utilization x total_HBM  =  weights  +  activation peak  +  KV cache
                                       (fixed)     (measured by a       (whatever
                                                    profiling run)       is left)
```

The order matters: weights load first, then a **dummy forward pass at maximum batch shape** measures
the activation peak, and the KV cache gets the remainder. Our vllm-ascend worker does this explicitly —
read `determine_available_memory` and find the subtraction:

In [4]:
show(VLLM_ASCEND, "vllm_ascend/worker/worker.py", r"def determine_available_memory", after=32)

--- /Users/sean/projects/ai/vllm-ascend/vllm_ascend/worker/worker.py:327 ---
  325  
  326      @torch.inference_mode()
  327      def determine_available_memory(self) -> int:
  328          """Profiles the peak memory usage of the model to determine how much
  329          memory can be used for KV cache without OOMs.
  330  
  331          The engine will first conduct a profiling of the existing memory usage.
  332          Then, it calculates the free memory that can be used for KV cache in
  333          bytes.
  334          """
  335          GiB = lambda b: b / GiB_bytes
  336  
  337          # Execute a forward pass with dummy inputs to profile the memory usage
  338          # of the model.
  339          with memory_profiling(
  340              self.init_snapshot,
  341              weights_memory=int(self.model_runner.model_memory_usage),
  342          ) as profile_result:
  343              self.model_runner.profile_run()
  344  
  345          free_gpu_memory = profile

This one function explains both of our A2 incidents:

- **`0.85` failed with \"not enough KV cache\"**: `requested_memory` (0.85 x HBM) minus weights minus the
  profiled peak left less than one `max-model-len=65536` sequence's worth of KV blocks. Raising to 0.90
  enlarged `requested_memory`; everything extra went straight to KV.
- **`0.94` passed init but OOM'd at runtime**: the profiling run *estimates* the activation peak. At 0.94
  the safety margin between the estimate and the true runtime peak was gone — real traffic exceeded the
  estimate and allocation failed *after* serving started. That's why ~0.90 is the practical ceiling:
  it is not a magic number, it is \"how much slack the profiler's estimate needs\".

### KV cache: the per-token price

KV is stored in fixed-size **blocks** (pages) of `block_size` tokens. The price per page for standard
attention is in `AttentionSpec.real_page_size_bytes` — read it, then we'll turn it into a formula:

In [6]:
# Standard attention: 2 (K and V) x block_size x kv_heads x head_size x dtype_size
show(VLLM, "vllm/v1/kv_cache_interface.py", r"def real_page_size_bytes", after=10)
# MLA (DeepSeek) overrides this — find the second definition and see what changed:
show(VLLM, "vllm/v1/kv_cache_interface.py", r"class MLAAttentionSpec", after=20)

--- /Users/sean/projects/ai/vllm/vllm/v1/kv_cache_interface.py:80 ---
   78  
   79      @property
   80      def real_page_size_bytes(self) -> int:
   81          return (
   82              2
   83              * self.block_size
   84              * self.num_kv_heads
   85              * self.head_size
   86              * get_dtype_size(self.dtype)
   87          )
   88  
   89  
--- /Users/sean/projects/ai/vllm/vllm/v1/kv_cache_interface.py:191 ---
  189  
  190  @dataclass(frozen=True, kw_only=True)
  191  class MLAAttentionSpec(FullAttentionSpec):
  192      # TODO(Lucas/Chen): less hacky way to do this
  193      cache_dtype_str: str | None = None
  194  
  195      @property
  196      def real_page_size_bytes(self) -> int:
  197          if self.cache_dtype_str == "fp8_ds_mla":
  198              # See `vllm/v1/attention/backends/mla/flashmla_sparse.py`
  199              #  for details.
  200              return self.block_size * 656
  201          return (
  202            

In [7]:
# KV bytes per token, from the formulas above.
def kv_bytes_per_token(num_layers, num_kv_heads, head_size, dtype_bytes, mla=False):
    per_layer = num_kv_heads * head_size * dtype_bytes
    if not mla:
        per_layer *= 2          # K and V; MLA stores one compressed latent instead
    return num_layers * per_layer

# Worked example with PUBLIC DeepSeek-V3 MLA dims (kv_lora_rank=512 + qk_rope_head_dim=64 -> head_size 576,
# 61 layers, 1 'kv head'). V4-Flash differs (hybrid layers, fewer full-attention layers) — EXERCISE:
# pull config.json from a node and redo this with real V4-Flash numbers.
v3 = kv_bytes_per_token(num_layers=61, num_kv_heads=1, head_size=576, dtype_bytes=2, mla=True)
print(f"DeepSeek-V3-style MLA: {v3 / 1024:.1f} KiB/token -> a 65536-token sequence = {v3 * 65536 / 2**30:.2f} GiB")

# Contrast: a hypothetical 61-layer model with standard GQA (8 kv heads x 128 dim):
gqa = kv_bytes_per_token(num_layers=61, num_kv_heads=8, head_size=128, dtype_bytes=2)
print(f"Same depth, GQA-8:     {gqa / 1024:.1f} KiB/token ({gqa / v3:.1f}x more) — this is what MLA buys")

DeepSeek-V3-style MLA: 68.6 KiB/token -> a 65536-token sequence = 4.29 GiB
Same depth, GQA-8:     244.0 KiB/token (3.6x more) — this is what MLA buys


Note for V4-Flash: it is a **hybrid** model — only some layers are full attention, others are cheaper
(see `SlidingWindowSpec` / other specs in the same file, and our `--no-disable-hybrid-kv-cache-manager`
flag, which lets vLLM size those layer groups differently instead of paying full-attention price everywhere).

**Check yourself**
1. Why does raising `max-model-len` raise the *minimum* KV memory needed, even with no traffic?
2. Activation peak scales with `max-num-batched-tokens`. Given the equation at the top, explain the
   tuning comment we left in `run_prefill_node0.sh`: \"lower --max-num-batched-tokens (8192 -> 4096)
   instead — it drives the activation peak, freeing that memory for KV cache.\"

## 3. Scheduler knobs: spending a token budget

Every step, the scheduler gets a budget and spends it across requests. The three knobs:

| Flag | Meaning | vLLM default |
|------|---------|--------------|
| `--max-model-len` | longest sequence (prompt + output) a request may reach | model's own limit |
| `--max-num-seqs` | max requests *running concurrently* | 128 |
| `--max-num-batched-tokens` | max **tokens processed per engine step**, summed over all requests | 2048 |

(Defaults verified in `vllm/config/scheduler.py` — `DEFAULT_MAX_NUM_BATCHED_TOKENS = 2048`, `DEFAULT_MAX_NUM_SEQS = 128`.)

**Chunked prefill** falls out of the budget naturally: a 64K prompt with an 8192 budget is simply
scheduled 8192 tokens at a time, 8 steps in a row, decodes interleaving in between. Read the actual
budget loop — `token_budget` decrements as requests are admitted:

In [8]:
show(VLLM, "vllm/v1/core/sched/scheduler.py", r"token_budget = self.max_num_scheduled_tokens", before=5, after=30)

--- /Users/sean/projects/ai/vllm/vllm/v1/core/sched/scheduler.py:357 ---
  352          scheduled_running_reqs: list[Request] = []
  353          preempted_reqs: list[Request] = []
  354  
  355          req_to_new_blocks: dict[str, KVCacheBlocks] = {}
  356          num_scheduled_tokens: dict[str, int] = {}
  357          token_budget = self.max_num_scheduled_tokens
  358          if self._pause_state == PauseState.PAUSED_ALL:
  359              # Do not schedule any requests when paused.
  360              token_budget = 0
  361  
  362          # Encoder-related.
  363          scheduled_encoder_inputs: dict[str, list[int]] = {}
  364          encoder_compute_budget = self.max_num_encoder_input_tokens
  365          # Spec decode-related.
  366          scheduled_spec_decode_tokens: dict[str, list[int]] = {}
  367  
  368          # For logging.
  369          scheduled_timestamp = time.monotonic()
  370  
  371          self.kv_cache_manager.new_step_starts()
  372  
  373         

In [9]:
# Toy simulator of the budget loop above: watch a 64K prompt chunk through while decodes keep flowing.
def simulate(prompt_lens, max_num_batched_tokens, max_num_seqs, steps=12, spec_tokens=0):
    reqs = [{"id": i, "todo": n, "decoding": n == 0} for i, n in enumerate(prompt_lens)]
    per_decode = 1 + spec_tokens
    for step in range(1, steps + 1):
        budget, sched = max_num_batched_tokens, []
        for r in reqs[:max_num_seqs]:
            n = per_decode if r["decoding"] else min(r["todo"], budget)
            if budget < n or n <= 0:
                continue
            budget -= n
            sched.append(f"req{r['id']}:{'D' if r['decoding'] else 'P'}{n}")
            if not r["decoding"]:
                r["todo"] -= n
                if r["todo"] == 0:
                    r["decoding"] = True
        print(f"step {step:2d}: used {max_num_batched_tokens - budget:5d}/{max_num_batched_tokens}  {' '.join(sched)}")

print("== our A2 prefill engine shape: 8192 budget, 4 seqs, one 64K prompt + three short ==")
simulate([65536, 300, 300, 300], max_num_batched_tokens=8192, max_num_seqs=4)

== our A2 prefill engine shape: 8192 budget, 4 seqs, one 64K prompt + three short ==
step  1: used  8192/8192  req0:P8192
step  2: used  8192/8192  req0:P8192
step  3: used  8192/8192  req0:P8192
step  4: used  8192/8192  req0:P8192
step  5: used  8192/8192  req0:P8192
step  6: used  8192/8192  req0:P8192
step  7: used  8192/8192  req0:P8192
step  8: used  8192/8192  req0:P8192
step  9: used   901/8192  req0:D1 req1:P300 req2:P300 req3:P300
step 10: used     4/8192  req0:D1 req1:D1 req2:D1 req3:D1
step 11: used     4/8192  req0:D1 req1:D1 req2:D1 req3:D1
step 12: used     4/8192  req0:D1 req1:D1 req2:D1 req3:D1


### Why our prefill and decode engines pick opposite values

| | prefill engine | decode engine |
|---|---|---|
| `max-num-batched-tokens` | **8192** — big chunks, compute-bound work | **144** — decodes are 3 tokens each |
| `max-num-seqs` | **4** — few huge prompts at a time | **48** — many streams generating |

The decode `144` is not arbitrary: with MTP speculative decoding (`num_speculative_tokens: 2`) each
sequence schedules 1 + 2 = 3 tokens per step, and **48 seqs x 3 = 144** — the budget exactly fits a full
house, and the cudagraph is captured at that one shape (`cudagraph_capture_sizes: [144]`).
Run the simulator with the decode shape and confirm the budget never starves anyone:

In [14]:
print("== decode engine shape: 48 already-decoding seqs, 3 tokens each (MTP k=2) ==")
simulate([0] * 48, max_num_batched_tokens=144, max_num_seqs=48, steps=3, spec_tokens=2)

== decode engine shape: 48 already-decoding seqs, 3 tokens each (MTP k=2) ==
step  1: used   144/144  req0:D3 req1:D3 req2:D3 req3:D3 req4:D3 req5:D3 req6:D3 req7:D3 req8:D3 req9:D3 req10:D3 req11:D3 req12:D3 req13:D3 req14:D3 req15:D3 req16:D3 req17:D3 req18:D3 req19:D3 req20:D3 req21:D3 req22:D3 req23:D3 req24:D3 req25:D3 req26:D3 req27:D3 req28:D3 req29:D3 req30:D3 req31:D3 req32:D3 req33:D3 req34:D3 req35:D3 req36:D3 req37:D3 req38:D3 req39:D3 req40:D3 req41:D3 req42:D3 req43:D3 req44:D3 req45:D3 req46:D3 req47:D3
step  2: used   144/144  req0:D3 req1:D3 req2:D3 req3:D3 req4:D3 req5:D3 req6:D3 req7:D3 req8:D3 req9:D3 req10:D3 req11:D3 req12:D3 req13:D3 req14:D3 req15:D3 req16:D3 req17:D3 req18:D3 req19:D3 req20:D3 req21:D3 req22:D3 req23:D3 req24:D3 req25:D3 req26:D3 req27:D3 req28:D3 req29:D3 req30:D3 req31:D3 req32:D3 req33:D3 req34:D3 req35:D3 req36:D3 req37:D3 req38:D3 req39:D3 req40:D3 req41:D3 req42:D3 req43:D3 req44:D3 req45:D3 req46:D3 req47:D3
step  3: used   144/144  req0

**Check yourself**
1. On the decode engine, what happens to per-token latency if you set `max-num-seqs 96` but leave the
   token budget at 144? (Re-run the simulator to verify your prediction.)
2. Our PD decode engines re-prefill the *whole prompt locally* if KV transfer fails. At 144 tokens/step,
   how many steps does a 32K prompt cost? (This was hypothesis H1 in our throughput investigation.)

## 4. Parallelism: TP, DP, EP, PP

Four ways to use more devices — they shard *different things* and pay *different communication*:

| | shards what | adds what communication | when it helps |
|---|---|---|---|
| **TP** (tensor) | every weight matrix, within a layer | all-reduce **per layer, per step** | model too big for one device; latency-sensitive if links are fast |
| **PP** (pipeline) | whole layers, into stages | activations between stages | very deep models; adds pipeline bubbles |
| **DP** (data) | nothing — full replica per rank | none for dense models; lockstep for MoE | throughput scaling; each rank schedules its own requests |
| **EP** (expert) | MoE expert weights across ranks | **all-to-all** (dispatch/combine) per MoE layer | MoE models — the only way expert memory shrinks |

The config fields live in `ParallelConfig`:

In [15]:
show(VLLM, "vllm/config/parallel.py", r"tensor_parallel_size: int", before=8, after=40)

--- /Users/sean/projects/ai/vllm/vllm/config/parallel.py:103 ---
   95  
   96  
   97  @config
   98  class ParallelConfig:
   99      """Configuration for the distributed execution."""
  100  
  101      pipeline_parallel_size: int = 1
  102      """Number of pipeline parallel groups."""
  103      tensor_parallel_size: int = 1
  104      """Number of tensor parallel groups."""
  105      prefill_context_parallel_size: int = 1
  106      """Number of prefill context parallel groups."""
  107      data_parallel_size: int = 1
  108      """Number of data parallel groups. MoE layers will be sharded according to
  109      the product of the tensor parallel size and data parallel size."""
  110      data_parallel_size_local: int = 1
  111      """Number of local data parallel groups."""
  112      data_parallel_rank: int = 0
  113      """Rank of the data parallel group."""
  114      data_parallel_rank_local: int | None = None
  115      """Local rank of the data parallel group,
  116  

### DP is not what it looks like (for MoE)

For a dense model, DP ranks are fully independent engines. For an **MoE model with
`--enable-expert-parallel`, the EP group spans the whole DP x TP world** — experts are sharded across
*all* ranks, so every MoE layer's all-to-all involves every rank. Consequence: DP ranks must run their
forward passes **in lockstep**. If one rank has no requests, it cannot just idle — it must join the
collectives with a fake batch. vLLM literally does this:

In [16]:
# An idle DP rank executes dummy passes so the EP all-to-all doesn't deadlock:
show(VLLM, "vllm/v1/engine/core.py", r"must execute a dummy pass", before=8, after=8)

--- /Users/sean/projects/ai/vllm/vllm/v1/engine/core.py:1705 ---
 1697              self._maybe_publish_request_counts()
 1698  
 1699              local_unfinished_reqs = self.scheduler.has_unfinished_requests()
 1700              if not executed:
 1701                  if not local_unfinished_reqs and not self.engines_running:
 1702                      # All engines are idle.
 1703                      continue
 1704  
 1705                  # We are in a running state and so must execute a dummy pass
 1706                  # if the model didn't execute any ready requests.
 1707                  self.execute_dummy_batch()
 1708  
 1709              # 3) All-reduce operation to determine global unfinished reqs.
 1710              self.engines_running = self._has_global_unfinished_reqs(
 1711                  local_unfinished_reqs
 1712              )


In [18]:
# Why our 1P2D layout exists: EP size = DP x TP, and expert weights shard across the EP group.
def expert_gib_per_npu(total_expert_gib, dp, tp):
    ep = dp * tp
    return total_expert_gib / ep

TOTAL = 400  # placeholder GiB of routed-expert weights — substitute the real V4-Flash number
for name, dp, tp in [("pd-2p2d prefill (per node)", 8, 1), ("pd-1p2d prefill (2 nodes)", 16, 1), ("a3/1node", 4, 4)]:
    print(f"{name:28s} EP={dp * tp:3d} -> {expert_gib_per_npu(TOTAL, dp, tp):6.1f} GiB experts per NPU")
print("\nDoubling EP halves per-NPU expert memory -> that memory becomes KV cache (chapter 2).")
print("The price: the all-to-all now crosses nodes. Prefill tolerates that (throughput-bound);")
print("decode pays it on every generated token (TPOT) — which is why 1P2D merges only the prefillers.")

pd-2p2d prefill (per node)   EP=  8 ->   50.0 GiB experts per NPU
pd-1p2d prefill (2 nodes)    EP= 16 ->   25.0 GiB experts per NPU
a3/1node                     EP= 16 ->   25.0 GiB experts per NPU

Doubling EP halves per-NPU expert memory -> that memory becomes KV cache (chapter 2).
The price: the all-to-all now crosses nodes. Prefill tolerates that (throughput-bound);
decode pays it on every generated token (TPOT) — which is why 1P2D merges only the prefillers.


### Internal vs external DP — three deployment shapes we actually run

1. **Internal DP, one node** (`a2/pd-2p2d` engines): one `vllm serve`, `--data-parallel-size 8
   --data-parallel-size-local 8`, one API server, engine-internal load balancing across ranks.
2. **Internal DP, multi-node** (`a2/pd-1p2d` prefill, `a2/2nodes`): a leader plus `--headless` followers
   (`--data-parallel-start-rank N`); still ONE API server, on the leader.
3. **External DP** (`a3/pd-1p1d-extdp`, official `examples/external_online_dp/`): one `vllm serve`
   *process per DP rank*, each with `--data-parallel-rank i` and **its own API server**; a proxy
   load-balances across all of them. Same collectives, but scheduling moves out of the engine.

**Check yourself**
1. Why does killing one node of a 2-node internal-DP engine kill the whole engine, while our 2P2D
   layout only loses half its capacity?
2. TP=4 on `a3/1node` keeps each TP group on one node. What would TP=16 across two A2 nodes cost,
   and per what unit? (Hint: row 1 of the table — per layer, per step.)

## 5. Our fleet, annotated

The cell below parses every `run_*.sh` in `deepseekv4/flash/` and tabulates the flags you now know.
Read a column, explain every row from chapters 2–4 — that's the exam.

In [ ]:
import json
FLAGS = ["data-parallel-size", "data-parallel-size-local", "data-parallel-start-rank", "tensor-parallel-size",
         "max-model-len", "max-num-batched-tokens", "max-num-seqs", "gpu-memory-utilization", "port"]

def parse_run_script(path):
    # strip comment lines first -- our script headers mention flags in prose
    text = "\n".join(l for l in path.read_text(encoding="utf-8").splitlines()
                     if not l.lstrip().startswith("#"))
    row = {}
    for f in FLAGS:
        m = re.search(rf"--{f}\s+(\S+)", text)
        row[f] = m.group(1).strip('"') if m else ""
    row["headless"] = "yes" if "--headless" in text else ""
    row["EP"] = "yes" if "--enable-expert-parallel" in text else ""
    m = re.search(r"--kv-transfer-config\s+'(\{.*?\})'", text, re.S)
    if m:
        kv = json.loads(m.group(1))
        row["kv_role"] = kv.get("kv_role", "").replace("kv_", "")
        row["kv_port"] = kv.get("kv_port", "")
    return row

scripts = sorted((REPO / "deepseekv4/flash").rglob("run_*.sh"))
scripts = [s for s in scripts if "proxy" not in s.name and "template" not in s.name]
cols = FLAGS + ["headless", "EP", "kv_role", "kv_port"]
namew = max(len(str(s.relative_to(REPO / 'deepseekv4/flash'))) for s in scripts)
print(f"{'script':<{namew}}  " + "  ".join(c[:12] for c in cols))
for s in scripts:
    row = parse_run_script(s)
    print(f"{str(s.relative_to(REPO / 'deepseekv4/flash')):<{namew}}  " + "  ".join(f"{row.get(c, ''):<{max(len(c[:12]), 4)}}" for c in cols))

### Anatomy of `--kv-transfer-config` (PD disaggregation)

```json
{"kv_connector": "MooncakeHybridConnector",   // which transfer implementation; 'Hybrid' = knows V4-Flash's mixed layer types
 "kv_role": "kv_producer",                    // producer = prefill, consumer = decode
 "kv_port": "36000",                          // side-channel base port; occupies kv_port .. kv_port+num_NPUs.
                                                //   A3 gotcha: AscendDirectTransport reserves [20000, 20000+NPUs*1000)
                                                //   -> on 16-NPU A3 nodes kv_port must be >= 36000
 "engine_id": "0",                            // unique per engine across the deployment
 "kv_connector_extra_config": {                // each side must know the OTHER side's parallel layout
    "prefill": {"dp_size": 16, "tp_size": 1},  //   to map which remote worker holds which KV blocks
    "decode":  {"dp_size": 8,  "tp_size": 1}}}
```

Source: `vllm_ascend/distributed/kv_transfer/kv_p2p/mooncake_hybrid_connector.py` —
`_get_prefill_decode_size` reads the extra config (and notably has **no** assert that prefill and decode
dp_size match, which is what makes our asymmetric 1P2D layout legal).

## 6. Pointers for later (each is a future session)

| Topic | Our flag | Where to read |
|---|---|---|
| MTP speculative decoding | `--speculative-config '{"method": "deepseek_mtp", ...}'` | `vllm/v1/spec_decode/`, vllm-ascend MTP patches (`git log --grep=mtp` in the v4 branch) |
| w8a8 quantization | `--quantization ascend` | `vllm_ascend/quantization/methods/w8a8_static.py` |
| Graph capture modes | `--compilation-config '{"cudagraph_mode": "FULL_DECODE_ONLY"}'`, `--enforce-eager` | `vllm_ascend/platform.py` (search `FULL_DECODE_ONLY`) — also explains why decode captures at size 144 |
| FlashComm / sequence parallelism | `VLLM_ASCEND_ENABLE_FLASHCOMM1` | `vllm_ascend/utils.py` (search `enable_sp`) |
| Hybrid KV cache manager | `--no-disable-hybrid-kv-cache-manager` | `vllm/v1/kv_cache_interface.py` specs + `vllm_ascend/patch/platform/patch_core.py` |
| The PD proxy internals | `run_proxy.sh` | `load_balance_proxy_server_example.py` in any of our pd-* dirs (928 lines, chapter-5 knowledge assumed) |

**A study habit that works with this repo:** every time a deployment misbehaves, find the *one function*
that made the decision (this notebook showed you the main ones), read it, then write the fix. That is
exactly how we found the 0.94 OOM, the group-0 proxy routing, and the kv_port reservation bug."